# NDSI over Pyrenees in Jupytergis with `jupyter-tiler`

To run this notebook, you'll need the dependencies in this repository's `"play"` dependency group.
For most people, installing the right dependencies might look like:

```bash
uv sync --group dev --group test --group play
```

## First, grab some data and do a calculation on it

We want to see that we can display calculated data from an in-memory dataset.

### Get Sentinel 2 data as an Xarray `DataSet`

#### Define a Pyrenees Bounding Box

In [ ]:
from leafmap.plot import bbox_to_gdf

bbox = [-1.7, 42.2, 3.0, 43.4]
bbox_to_gdf(bbox).explore()

#### Query the Element84 Earth Search STAC catalog for Sentinel2 data

In [ ]:
from pystac_client import Client

sentinel2_stac_items = (
    Client.open("https://earth-search.aws.element84.com/v1")
    .search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime="2024-04-15/2024-06-15",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    .item_collection()
)
sentinel2_stac_items

#### Load the data into an Xarray `Dataset`

In [ ]:
import odc.stac

sentinel2_dataset = odc.stac.load(
    sentinel2_stac_items,
    bands=["red", "green", "blue", "swir16"],
    chunks={'x': 2048, 'y': 2048},
    bbox=bbox,
    resolution=20,
    groupby="solar_day",
)
sentinel2_dataset

In [ ]:
config_options = {
                  'AWS_VIRTUAL_HOSTING': 'FALSE',
                  'GDAL_HTTP_UNSAFESSL': 'YES',
                  'AWS_HTTPS': 'YES',
                  'AWS_NO_SIGN_REQUEST': 'YES'}
import rasterio

## Dask

In [ ]:
#cluster.close()

In [ ]:
from dask_kubernetes.operator import KubeCluster

config = {
   "name": "ndsi-jgis",
   "namespace": "jhub",
   "image": "ghcr.io/geojupyter/presentation-cng2026:sha-723a7c3",
   "n_workers": 2,
   "resources":{"requests": {"memory": "4Gi"}, "limits": {"memory": "4Gi"}}
}

cluster = KubeCluster(**config)

In [ ]:
cluster.scale(20)
cluster

In [ ]:
dask_client = cluster.get_client()
dask_client

In [ ]:
def configure_workers_environment():
    import os
    for key, value in config_options.items():
        os.environ[key] = value
        
    rio_env = rasterio.Env(**config_options)
    rio_env.__enter__()
    return None
    
dask_client.register_worker_callbacks(configure_workers_environment)

### Visualize raw data in RGB

In [ ]:
%%time
with rasterio.Env(**config_options):
    rgb = sentinel2_dataset[["red", "green", "blue"]].to_array(dim="band").max("time")
    rgb_scaled = (rgb / 3000).clip(0, 1)  # Scale and clip for display
    rgb_scaled

In [ ]:
#Visualize subset
with rasterio.Env(**config_options):
    rgb_scaled[:,5000:6000,10000:12000].plot.imshow()

### Calculate NDSI

In [ ]:
%%time
with rasterio.Env(**config_options):
    # Filter on red band value
    sentinel2_dataset_red_high = sentinel2_dataset.where(sentinel2_dataset.red.mean(dim="time") > 200)
    
    ndsi = (
        (
            (sentinel2_dataset_red_high.green - sentinel2_dataset_red_high.swir16)
            / (sentinel2_dataset_red_high.green + sentinel2_dataset_red_high.swir16)
        )
        .max(dim="time")
        .where(lambda ndsi: ndsi < 1)
    )

    #PReload everything, avoid multiple get and bandwith overload
    ndsi = ndsi.persist()
    

In [ ]:
ndsi[5000:6000,10000:12000].plot.imshow()

## Test out `jupyter-tiler`

...with the `ndsi` `DataArray` we just calculated!

In [ ]:
type(ndsi)

In [ ]:
from jupytergis import GISDocument
doc = GISDocument(
    longitude=1.7092461496028497,
    latitude=42.530360648396055,
    zoom=11.38992197518327,
)
await doc.ready()
doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png"
)
doc.sidecar(title="Map", anchor="split-right")

In [ ]:
await doc.add_data_array_layer(
    name="NDSI 2024",
    data_array=ndsi,
    colormap_name="viridis",
    colormap_range=(0, 1),
)

In [ ]:
snow = ndsi > 0.4
snow = snow.astype('uint8')
snow

In [ ]:
await doc.add_data_array_layer(
    name="Snow Cover 2024",
    data_array=snow,
    colormap_range=(0, 1),
)

## Add another year

In [ ]:
sentinel2_stac_items = (
    Client.open("https://earth-search.aws.element84.com/v1")
    .search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime="2022-04-15/2022-06-15",
        query={"eo:cloud_cover": {"lt": 10}}, #Cloudy month? But too much cloud is wrong
    )
    .item_collection()
)
sentinel2_stac_items

In [ ]:
sentinel2_dataset = odc.stac.load(
    sentinel2_stac_items,
    bands=["red", "green", "blue", "swir16"],
    chunks={'x': 2048, 'y': 2048},
    bbox=bbox,
    resolution=20,
    groupby="solar_day",
)
sentinel2_dataset

In [ ]:
%%time
with rasterio.Env(**config_options):
    # Filter on red band value
    sentinel2_dataset_red_high = sentinel2_dataset.where(sentinel2_dataset.red.mean(dim="time") > 200)
    ndsi_2022 = (
        (
            (sentinel2_dataset_red_high.green - sentinel2_dataset_red_high.swir16)
            / (sentinel2_dataset_red_high.green + sentinel2_dataset_red_high.swir16)
        )
        .max(dim="time")
        .where(lambda ndsi_2022: ndsi_2022 < 1)
    )

    #PReload everything, avoid multiple get and bandwith overload
    ndsi_2022 = ndsi_2022.persist()

In [ ]:
snow_2022 = ndsi_2022 > 0.4
snow_2022 = snow_2022.astype('uint8')
snow_2022

In [ ]:
await doc.add_data_array_layer(
    name="NDSI 2022",
    data_array=ndsi_2022,
    colormap_name="cool",
    colormap_range=(0, 1),
)

In [ ]:
await doc.add_data_array_layer(
    name="Snow Cover 2022",
    data_array=snow_2022,
    colormap_range=(0, 1),
    colormap_name="cool"
)

### Build a temporal STAC

In [ ]:
cluster.close()